In [23]:
import glob
import re
import pandas as pd
from unidecode import unidecode
from polyfuzz import PolyFuzz
from polyfuzz.models import RapidFuzz, TFIDF, EditDistance, Embeddings
from flair.embeddings import TransformerWordEmbeddings, WordEmbeddings
from jellyfish import jaro_winkler_similarity

In [ ]:
class MatchFactory:

    def __init__(self):
        self.factory = {
            'exact': self._exact_matcher,
            'acronym': self._acronym_matcher,
            'short': self._short_matcher,
            'fuzzy': self._fuzzy_matcher
            }
        return

    def match_factory(self, from_list, to_list):
        self.from_list = from_list
        self.to_list = to_list
        match = {}
        for match_type in self.factory.keys():
            print(f'{len(from_list) = } {len(to_list) = }')
            match  |= self.factory[match_type](from_list, to_list)
            print(match)
            from_list = [from_name for from_name in from_list if from_name not in match.keys()]
            to_list = [to_name for to_name in to_list if to_name not in match.values()]
            if not from_list or not to_list:
                return match
        return match

    def _exact_matcher(self, from_list, to_list):
        exact_match = []
        exact_match.extend(item for item in from_list if item in to_list)
        print(f'{len(exact_match) = }/{len(from_list) = }')
        return dict(zip(exact_match, exact_match))

    def _acronym_matcher(self, from_list, to_list):
        acronym_match = {}
        for from_name in from_list:
            if m := re.search(r'([A-Z]+?){3,}', from_name):
                match = m[0]
                if match not in ['USA', 'UK']:
                    print(f'{from_name = } {match = }')
                    for to_name in to_list:
                        if match == to_name or match in to_name:
                            print(f'== {from_name = } {match = } {to_name = }')
                            acronym_match[from_name] = to_name
        return acronym_match

    def _short_matcher(self, from_list, to_list):
        to_dict = dict(zip(to_list, self.tidy_list(to_list)))
        short_match = {}
        for from_name, from_short in zip(from_list, self.tidy_list(from_list)):
            print(f'{from_name = } {from_short = }')
            for to_name, to_short in to_dict.items():
                print(f'{to_name = } {to_short = }')
                if from_short in to_short:
                    short_match[from_name] = to_name
                    break
        return short_match

    def _fuzzy_matcher(self, from_list, to_list):
        tfidf = TFIDF(n_gram_range=(3,3), min_similarity=0.5, model_id="TF-IDF")
        rapid_fuzz = RapidFuzz(n_jobs=1, score_cutoff=0.8, model_id='RapidFuzz')
        matchers = [tfidf, rapid_fuzz]
        model = PolyFuzz(matchers)
        model.match(from_list, to_list) #self.tidy_list(from_list), to_list)
        match = pd.concat(list(model.get_matches().values()), axis=0).sort_values('Similarity', ascending=False).drop_duplicates(subset=['From'])
        print(match.tail(32))
        return {f: t for f, t, s in zip(match.From, match.To, match.Similarity) if s > 0.8 and t != None}

    def tidy_list(self, in_list):
        out_list = []
        for item in in_list:
            parts = [part.strip().lower() for part in unidecode(item).split(' ')]
            out_list.append(' '.join([p for p in parts if p not in ['the', 'of', '&', 'and', 'de', '-', 'universite', 'universidade', 'universitat', 'university']])) #'university'
        return out_list

In [25]:
def main():
    mf = MatchFactory()
    match = mf.match_factory(from_list=['Hello', 'Dolly', 'The Dolly', 'Pete', 'BHP', 'Pete Dolly'], to_list=['Hello', 'Peter', 'Dolly Bird', 'BHP Pty Ltd', 'Hello BHP'])
    print(match)

In [26]:
if __name__ == "__main__":
    main()
    print("DONE!")

len(from_list) = 6 len(to_list) = 5
len(exact_match) = 1/len(from_list) = 6
{'Hello': 'Hello'}
len(from_list) = 5 len(to_list) = 4
from_name = 'BHP' match = 'BHP'
== from_name = 'BHP' match = 'BHP' to_name = 'BHP Pty Ltd'
== from_name = 'BHP' match = 'BHP' to_name = 'Hello BHP'
{'Hello': 'Hello', 'BHP': 'Hello BHP'}
len(from_list) = 4 len(to_list) = 3
from_name = 'Dolly' from_short = 'dolly'
to_name = 'Peter' to_short = 'peter'
to_name = 'Dolly Bird' to_short = 'dolly bird'
from_name = 'The Dolly' from_short = 'dolly'
to_name = 'Peter' to_short = 'peter'
to_name = 'Dolly Bird' to_short = 'dolly bird'
from_name = 'Pete' from_short = 'pete'
to_name = 'Peter' to_short = 'peter'
from_name = 'Pete Dolly' from_short = 'pete dolly'
to_name = 'Peter' to_short = 'peter'
to_name = 'Dolly Bird' to_short = 'dolly bird'
to_name = 'BHP Pty Ltd' to_short = 'bhp pty ltd'
{'Hello': 'Hello', 'BHP': 'Hello BHP', 'Dolly': 'Dolly Bird', 'The Dolly': 'Dolly Bird', 'Pete': 'Peter'}
len(from_list) = 1 len(to_